In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
orders_bronze = spark.table("retail_raw.bronze_orders")

display(orders_bronze.limit(5))

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,source_file,ingestion_ts,load_type
O0000001,2025-12-15 04:49:00,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000002,2026-03-13 03:29:00,C00713,P00768,S028,4,30049.26,0.1,102167.48,CARD,returned,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000003,2026-03-05 07:20:00,C02131,P00142,S072,5,59685.88,0.0,283507.93,COD,delivered,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000004,2025-12-28 13:56:00,C01399,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000005,2025-12-31 17:43:00,C00383,P00564,S049,2,46727.14,0.15,84108.85,COD,cancelled,orders_batch.csv,2026-07-09T19:08:32.990Z,batch


In [0]:
orders_bronze.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- gross_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



In [0]:
orders_quarantine = orders_bronze.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("product_id").isNull() |
    col("store_id").isNull()
)

display(orders_quarantine.limit(10))

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,source_file,ingestion_ts,load_type
O0000043,2025-11-13 12:37:00,null,P00543,S048,2,86349.23,0.0,164063.54,CARD,returned,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000127,2025-12-10 17:34:00,null,P00588,S029,2,unknown,0.1,bad_amount,NETBANKING,returned,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000263,2025-10-08 17:41:00,null,P00146,S016,1,7430.37,0.0,6687.33,COD,returned,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000345,2026-01-11 07:27:00,null,P00177,S038,2,2194.3,0.05,4169.17,UPI,cancelled,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000424,2025-11-15 16:05:00,null,P00071,S053,4,18907.54,0.0,68067.14,UPI,delivered,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000572,2025-12-27 01:37:00,null,P00069,S062,3,30411.38,0.1,86672.43,UPI,delivered,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000652,2025-12-01 16:23:00,null,P00644,S003,5,3769.83,0.15,16021.78,UPI,delivered,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0000808,2025-11-19 07:14:00,null,P00682,S054,3,33752.53,0.15,96194.71,CARD,shipped,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0001088,2026-02-09 10:09:00,null,P00090,S038,4,7578.09,0.15,27281.12,UPI,delivered,orders_batch.csv,2026-07-09T19:08:32.990Z,batch
O0001228,2026-02-24 01:48:00,null,P00129,S038,2,75613.95,0.15,151227.9,NETBANKING,delivered,orders_batch.csv,2026-07-09T19:08:32.990Z,batch


In [0]:
orders_valid = orders_bronze.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull() &
    col("product_id").isNotNull() &
    col("store_id").isNotNull()
)

In [0]:
orders_clean = (
    orders_valid
    .withColumn("order_ts", expr("try_cast(order_ts as timestamp)"))
    .withColumn("quantity", expr("try_cast(regexp_replace(quantity,'[^0-9]','') as int)"))
    .withColumn("unit_price", expr("try_cast(regexp_replace(unit_price,'[^0-9.]','') as double)"))
    .withColumn("discount_pct", expr("try_cast(regexp_replace(discount_pct,'[^0-9.]','') as double)"))
    .withColumn("gross_amount", expr("try_cast(regexp_replace(gross_amount,'[^0-9.]','') as double)"))
)

In [0]:
orders_clean.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



In [0]:
orders_invalid = orders_clean.filter(
    col("order_ts").isNull() |
    col("quantity").isNull() |
    col("unit_price").isNull() |
    col("gross_amount").isNull()
)

orders_clean_valid = orders_clean.filter(
    col("order_ts").isNotNull() &
    col("quantity").isNotNull() &
    col("unit_price").isNotNull() &
    col("gross_amount").isNotNull()
)

print("Invalid Clean Records:", orders_invalid.count())
print("Valid Clean Records:", orders_clean_valid.count())

Invalid Clean Records: 0
Valid Clean Records: 11675


In [0]:
window_order = Window.partitionBy("order_id").orderBy(col("ingestion_ts").desc())

orders_deduped = (
    orders_clean_valid
    .withColumn("rn", row_number().over(window_order))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Before Deduplication:", orders_clean_valid.count())
print("After Deduplication:", orders_deduped.count())

Before Deduplication: 11675
After Deduplication: 11498


In [0]:
orders_deduped.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.silver1_orders_clean")

In [0]:
orders_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.quarantine_orders_missing_ids")

In [0]:
spark.sql("""
SELECT 'silver1_orders_clean' AS table_name, COUNT(*) AS row_count
FROM retail_silver.silver1_orders_clean

UNION ALL

SELECT 'quarantine_orders_missing_ids', COUNT(*)
FROM retail_silver.quarantine_orders_missing_ids
""").show()

+--------------------+---------+
|          table_name|row_count|
+--------------------+---------+
|silver1_orders_clean|    11498|
|quarantine_orders...|       90|
+--------------------+---------+



In [0]:
customers_bronze = spark.table("retail_raw.bronze_customers")

display(customers_bronze.limit(5))

customer_id,customer_name,city,segment,gender,signup_date,status,source_file,ingestion_ts,load_type
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,customers_batch.csv,2026-07-09T19:08:51.698Z,batch
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,customers_batch.csv,2026-07-09T19:08:51.698Z,batch
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,customers_batch.csv,2026-07-09T19:08:51.698Z,batch
C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active,customers_batch.csv,2026-07-09T19:08:51.698Z,batch
C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive,customers_batch.csv,2026-07-09T19:08:51.698Z,batch


In [0]:
customers_bronze.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



In [0]:
customers_quarantine = customers_bronze.filter(
    col("customer_id").isNull()
)

customers_valid = customers_bronze.filter(
    col("customer_id").isNotNull()
)

In [0]:
customers_clean = (
    customers_valid
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", coalesce(trim(col("city")), lit("Unknown")))
    .withColumn("segment", coalesce(trim(col("segment")), lit("Unknown")))
    .withColumn("gender", coalesce(trim(col("gender")), lit("Unknown")))
    .withColumn("signup_date", expr("try_cast(signup_date as date)"))
    .withColumn("status", coalesce(trim(col("status")), lit("Unknown")))
)

In [0]:
customers_invalid_dates = customers_clean.filter(col("signup_date").isNull())

customers_clean_valid = customers_clean.filter(col("signup_date").isNotNull())

print("Missing Customer ID Records:", customers_quarantine.count())
print("Invalid Signup Date Records:", customers_invalid_dates.count())
print("Valid Customer Records:", customers_clean_valid.count())

Missing Customer ID Records: 0
Invalid Signup Date Records: 25
Valid Customer Records: 2535


In [0]:
customers_clean_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.silver1_customers_clean")

In [0]:
customers_invalid_dates.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.quarantine_customers_invalid_dates")

In [0]:
spark.sql("""
SELECT 'silver1_customers_clean' AS table_name, COUNT(*) AS row_count
FROM retail_silver.silver1_customers_clean

UNION ALL

SELECT 'quarantine_customers_invalid_dates', COUNT(*)
FROM retail_silver.quarantine_customers_invalid_dates
""").show()

+--------------------+---------+
|          table_name|row_count|
+--------------------+---------+
|silver1_customers...|     2535|
|quarantine_custom...|       25|
+--------------------+---------+



In [0]:
products_bronze = spark.table("retail_raw.bronze_products")

display(products_bronze.limit(5))

product_id,product_name,category,brand,unit_price,status,created_date,source_file,ingestion_ts,load_type
P00001,T-Shirt 1,Fashion,BrandC,unknown,discontinued,2025-02-15,products_batch.csv,2026-07-09T19:08:58.563Z,batch
P00002,Bedsheet 2,Home,BrandC,unknown,active,2024-04-20,products_batch.csv,2026-07-09T19:08:58.563Z,batch
P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23,products_batch.csv,2026-07-09T19:08:58.563Z,batch
P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07,products_batch.csv,2026-07-09T19:08:58.563Z,batch
P00005,Oil 5,null,BrandC,51796.39,active,2024-12-12,products_batch.csv,2026-07-09T19:08:58.563Z,batch


In [0]:
products_bronze.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



In [0]:
products_quarantine = products_bronze.filter(
    col("product_id").isNull()
)

products_valid = products_bronze.filter(
    col("product_id").isNotNull()
)

In [0]:
products_clean = (
    products_valid
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", coalesce(trim(col("category")), lit("Unknown")))
    .withColumn("brand", coalesce(trim(col("brand")), lit("Unknown")))
    .withColumn("unit_price", expr("try_cast(regexp_replace(unit_price,'[^0-9.]','') as double)"))
    .withColumn("status", coalesce(trim(col("status")), lit("Unknown")))
    .withColumn("created_date", expr("try_cast(created_date as date)"))
)

In [0]:
products_invalid = products_clean.filter(
    col("unit_price").isNull() |
    col("created_date").isNull()
)

products_clean_valid = products_clean.filter(
    col("unit_price").isNotNull() &
    col("created_date").isNotNull()
)

print("Missing Product ID Records:", products_quarantine.count())
print("Invalid Product Records:", products_invalid.count())
print("Valid Product Records:", products_clean_valid.count())

Missing Product ID Records: 0
Invalid Product Records: 20
Valid Product Records: 810


In [0]:
products_clean_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.silver1_products_clean")

In [0]:
products_invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.quarantine_products_invalid")

In [0]:
spark.sql("""
SELECT 'silver1_orders_clean' AS table_name, COUNT(*) AS row_count
FROM retail_silver.silver1_orders_clean

UNION ALL

SELECT 'silver1_customers_clean', COUNT(*)
FROM retail_silver.silver1_customers_clean

UNION ALL

SELECT 'silver1_products_clean', COUNT(*)
FROM retail_silver.silver1_products_clean
""").show()

+--------------------+---------+
|          table_name|row_count|
+--------------------+---------+
|silver1_orders_clean|    11498|
|silver1_customers...|     2535|
|silver1_products_...|      810|
+--------------------+---------+

